# Contract Intelligence Pipeline for Synthetic Financing Agreements

## Project Objective

The goal of this project is to transform unstructured PDF-based financing agreements into a structured dataset suitable for legal analytics and future machine learning applications.

This pipeline automatically extracts:

- Contract Type
- Borrower Information
- Lender Information
- Financing Amount
- Legal Clauses
- Jurisdiction Information
- Contractual Obligations

The extracted information is transformed into a structured dataset using Python and Pandas, enabling downstream analysis and intelligent contract processing.

---

## Technologies Used

- Python
- pdfplumber
- Pandas
- Regular Expressions / String Processing
- Jupyter Notebook

---

## Project Workflow

PDF Contracts

↓

Text Extraction

↓

Metadata Extraction

↓

Clause Extraction

↓

Obligation Extraction

↓

Structured Dataset

↓

CSV / Excel Export

# Dataset Overview

## Dataset Description

The dataset contains 30 synthetic financing agreements stored as PDF documents.

These contracts represent multiple financing categories including:

- Car Financing Agreements
- Health Financing Agreements
- Real Estate Financing Agreements
- Motorcycle Financing Agreements
- Student Loan Agreements

All contracts are text-based PDFs and contain legal clauses, party information, financing amounts, obligations, and jurisdiction details.



In [1]:
from pathlib import Path

for item in Path("../data").iterdir():
    print(item)

../data/.DS_Store
../data/raw-contracts
../data/processed


# Contract Discovery

## Purpose

Identify all available PDF contracts within the dataset.



In [2]:
from pathlib import Path

pdf_folder = Path("../data/raw-contracts")

print(pdf_folder.exists())

pdfs = list(pdf_folder.glob("*.pdf"))

print("Number of contracts:", len(pdfs))
print(pdfs[:5])

True
Number of contracts: 30
[PosixPath('../data/raw-contracts/Contract_16.pdf'), PosixPath('../data/raw-contracts/Contract_4.pdf'), PosixPath('../data/raw-contracts/Contract_5.pdf'), PosixPath('../data/raw-contracts/Contract_17.pdf'), PosixPath('../data/raw-contracts/Contract_7.pdf')]


# Text Extraction from PDF Documents

In [3]:
import pdfplumber

sample_pdf = pdfs[0]

print("Opening:", sample_pdf)

text = ""

with pdfplumber.open(sample_pdf) as pdf:
    for page in pdf.pages:
        page_text = page.extract_text()

        if page_text:
            text += page_text + "\n"

print(text[:5000])

Opening: ../data/raw-contracts/Contract_16.pdf
HEALTH FINANCING AGREEMENT
BORROWER: Julia Miller, residing at 123 Flower Street - New York, NY, holder of SSN nº
232854048.
LENDER: FINANCIAL BANK OF AMERICA Inc., registered under EIN nº 00-0000000,
headquartered at 1000 Main Avenue, New York, NY.
FINANCED AMOUNT: $42,751.00.
CLAUSE ONE – PURPOSE: This agreement aims to grant financing to the BORROWER for the
acquisition of the asset described in the preamble.
CLAUSE TWO – TERM: The financing period shall be up to 60 months, unless otherwise agreed
upon by the parties.
CLAUSE THREE – PAYMENT: The BORROWER shall pay monthly installments, increased by
interest of 1.2% per month and corrected annually by the Consumer Price Index (CPI).
CLAUSE FOUR – GUARANTEES: The financed asset will remain under fiduciary lien in favor of
the BANK until full settlement.
CLAUSE FIVE – INSURANCE: The BORROWER must contract mandatory insurance for the
financed asset, with minimum coverage as required by curr

In [4]:
lines = text.split("\n")
contract_type = lines[0].strip()
print(contract_type)

borrower_line = ""

for line in lines:
    if "BORROWER:" in line:
        borrower_line = line
        break
borrower = borrower_line.split("BORROWER:")[1].split(",")[0].strip()
print(borrower)

lender_line = ""
for line in lines:
    if "LENDER:" in line:
        lender_line = line
        break
lender = lender_line.split("LENDER:")[1].split(",")[0].strip()
print(lender)

for line in lines:
    if "FINANCED AMOUNT:" in line:
        amount_line = line
        break
amount = amount_line.split("FINANCED AMOUNT:")[1].strip()
print(amount)



HEALTH FINANCING AGREEMENT
Julia Miller
FINANCIAL BANK OF AMERICA Inc.
$42,751.00.


# Contract Metadata Extraction

In [5]:
contract_info = {
    "contract_type": contract_type,
    "borrower": borrower,
    "lender": lender,
    "amount": amount
}

print(contract_info)

{'contract_type': 'HEALTH FINANCING AGREEMENT', 'borrower': 'Julia Miller', 'lender': 'FINANCIAL BANK OF AMERICA Inc.', 'amount': '$42,751.00.'}


# Legal Clause Identification

In [6]:
clauses = []

for line in lines:
    if "CLAUSE" in line:
        clause_name = line.split("–")[1].split(":")[0].strip()
        clauses.append(clause_name)
clauses = list(dict.fromkeys(clauses))

print(clauses)


['PURPOSE', 'TERM', 'PAYMENT', 'GUARANTEES', 'INSURANCE', 'DEFAULT', 'TERMINATION', 'JURISDICTION']


In [7]:
# Updated Contract Info
contract_info["clauses"] = clauses

print(contract_info)

{'contract_type': 'HEALTH FINANCING AGREEMENT', 'borrower': 'Julia Miller', 'lender': 'FINANCIAL BANK OF AMERICA Inc.', 'amount': '$42,751.00.', 'clauses': ['PURPOSE', 'TERM', 'PAYMENT', 'GUARANTEES', 'INSURANCE', 'DEFAULT', 'TERMINATION', 'JURISDICTION']}


# Contractual Obligation Extraction

In [8]:
def extract_contract_info(text):

    lines = text.split("\n")

    contract_type = lines[0].strip()

    borrower = None
    lender = None
    amount = None

    for line in lines:

        if "BORROWER:" in line:
            borrower = line.split("BORROWER:")[1].split(",")[0].strip()

        if "LENDER:" in line:
            lender = line.split("LENDER:")[1].split(",")[0].strip()

        if "FINANCED AMOUNT:" in line:
            amount = line.split("FINANCED AMOUNT:")[1].strip()

    clauses = []

    for line in lines:

        if "CLAUSE" in line and "–" in line:

            try:
                clause_name = line.split("–")[1].split(":")[0].strip()
                clauses.append(clause_name)

            except:
                pass

    clauses = list(dict.fromkeys(clauses))

    jurisdiction = None
        
    for line in lines:
        if "Courts of" in line:
            jurisdiction = line.split("Courts of")[1].split(", to")[0].strip()
            break

    return {
        "contract_type": contract_type,
        "borrower": borrower,
        "lender": lender,
        "amount": amount,
        "clauses": clauses,
        "jurisdiction": jurisdiction
    }

In [9]:
contract_info = extract_contract_info(text)

print(contract_info)

{'contract_type': 'HEALTH FINANCING AGREEMENT', 'borrower': 'Julia Miller', 'lender': 'FINANCIAL BANK OF AMERICA Inc.', 'amount': '$42,751.00.', 'clauses': ['PURPOSE', 'TERM', 'PAYMENT', 'GUARANTEES', 'INSURANCE', 'DEFAULT', 'TERMINATION', 'JURISDICTION'], 'jurisdiction': 'New York, NY'}


# Dataset Construction

In [10]:
all_contracts = []

import pdfplumber

for pdf_file in pdfs:

    text = ""

    with pdfplumber.open(pdf_file) as pdf:

        for page in pdf.pages:

            page_text = page.extract_text()

            if page_text:
                text += page_text + "\n"

    contract_info = extract_contract_info(text)

    contract_info["file"] = pdf_file.name

    all_contracts.append(contract_info)

print("Contracts processed:", len(all_contracts))

Contracts processed: 30


In [11]:
# Converting into Dataframe
import pandas as pd

df = pd.DataFrame(all_contracts)

df.head()

,contract_type,borrower,lender,amount,clauses,jurisdiction,file
0,HEALTH FINANCING AGREEMENT,Julia Miller,FINANCIAL BANK OF AMERICA Inc.,"$42,751.00.","[PURPOSE, TERM, PAYMENT, GUARANTEES, INSURANCE...","New York, NY",Contract_16.pdf
1,MOTORCYCLE FINANCING AGREEMENT,Patricia Hall,FINANCIAL BANK OF AMERICA Inc.,"$36,160.00.","[PURPOSE, TERM, PAYMENT, GUARANTEES, INSURANCE...","New York, NY",Contract_4.pdf
2,HEALTH FINANCING AGREEMENT,Patricia Hall,FINANCIAL BANK OF AMERICA Inc.,"$40,267.00.","[PURPOSE, TERM, PAYMENT, GUARANTEES, INSURANCE...","New York, NY",Contract_5.pdf
3,STUDENT LOAN AGREEMENT,Richard Taylor,FINANCIAL BANK OF AMERICA Inc.,"$56,424.00.","[PURPOSE, TERM, PAYMENT, GUARANTEES, INSURANCE...","New York, NY",Contract_17.pdf
4,HEALTH FINANCING AGREEMENT,Mary Johnson,FINANCIAL BANK OF AMERICA Inc.,"$48,720.00.","[PURPOSE, TERM, PAYMENT, GUARANTEES, INSURANCE...","New York, NY",Contract_7.pdf


In [12]:
obligations = []

for line in lines:

    if "shall" in line.lower() or "must" in line.lower():

        obligations.append(line.strip())

for o in obligations:
    print(o)

CLAUSE TWO – TERM: The financing period shall be up to 60 months, unless otherwise agreed
CLAUSE THREE – PAYMENT: The BORROWER shall pay monthly installments, increased by
CLAUSE FIVE – INSURANCE: The BORROWER must contract mandatory insurance for the
CLAUSE TWO – TERM: The financing period shall be up to 60 months, unless otherwise agreed
CLAUSE THREE – PAYMENT: The BORROWER shall pay monthly installments, increased by
CLAUSE FIVE – INSURANCE: The BORROWER must contract mandatory insurance for the
CLAUSE TWO – TERM: The financing period shall be up to 60 months, unless otherwise agreed
CLAUSE THREE – PAYMENT: The BORROWER shall pay monthly installments, increased by
CLAUSE FIVE – INSURANCE: The BORROWER must contract mandatory insurance for the


In [13]:
from pathlib import Path

for file in Path("../data").rglob("*"):
    print(file)

../data/.DS_Store
../data/raw-contracts
../data/processed
../data/raw-contracts/Contract_16.pdf
../data/raw-contracts/Contract_4.pdf
../data/raw-contracts/Contract_5.pdf
../data/raw-contracts/Contract_17.pdf
../data/raw-contracts/Contract_7.pdf
../data/raw-contracts/Contract_15.pdf
../data/raw-contracts/Contract_29.pdf
../data/raw-contracts/Contract_28.pdf
../data/raw-contracts/Contract_14.pdf
../data/raw-contracts/Contract_6.pdf
../data/raw-contracts/Contract_2.pdf
../data/raw-contracts/Contract_10.pdf
../data/raw-contracts/Contract_11.pdf
../data/raw-contracts/Contract_3.pdf
../data/raw-contracts/Contract_13.pdf
../data/raw-contracts/Contract_1.pdf
../data/raw-contracts/Contract_12.pdf
../data/raw-contracts/Contract_23.pdf
../data/raw-contracts/Contract_22.pdf
../data/raw-contracts/Contract_20.pdf
../data/raw-contracts/Contract_21.pdf
../data/raw-contracts/Contract_19.pdf
../data/raw-contracts/Contract_25.pdf
../data/raw-contracts/Contract_30.pdf
../data/raw-contracts/Contract_24.pdf

In [14]:
# Data Understanding
df["contract_type"].value_counts()

contract_type
CAR FINANCING AGREEMENT            10
HEALTH FINANCING AGREEMENT          9
REAL ESTATE FINANCING AGREEMENT     5
MOTORCYCLE FINANCING AGREEMENT      4
STUDENT LOAN AGREEMENT              2
Name: count, dtype: int64

In [15]:
df.columns

Index(['contract_type', 'borrower', 'lender', 'amount', 'clauses',
       'jurisdiction', 'file'],
      dtype='object')

In [16]:
df.describe(include="all")

,contract_type,borrower,lender,amount,clauses,jurisdiction,file
count,30,30,30,30,30,30,30
unique,5,10,1,30,1,1,30
top,CAR FINANCING AGREEMENT,Julia Miller,FINANCIAL BANK OF AMERICA Inc.,"$42,751.00.","[PURPOSE, TERM, PAYMENT, GUARANTEES, INSURANCE...","New York, NY",Contract_16.pdf
freq,10,7,30,1,30,30,1


In [17]:
for line in lines:
    if "shall" in line.lower() or "must" in line.lower():
        print(line)

CLAUSE TWO – TERM: The financing period shall be up to 60 months, unless otherwise agreed
CLAUSE THREE – PAYMENT: The BORROWER shall pay monthly installments, increased by
CLAUSE FIVE – INSURANCE: The BORROWER must contract mandatory insurance for the
CLAUSE TWO – TERM: The financing period shall be up to 60 months, unless otherwise agreed
CLAUSE THREE – PAYMENT: The BORROWER shall pay monthly installments, increased by
CLAUSE FIVE – INSURANCE: The BORROWER must contract mandatory insurance for the
CLAUSE TWO – TERM: The financing period shall be up to 60 months, unless otherwise agreed
CLAUSE THREE – PAYMENT: The BORROWER shall pay monthly installments, increased by
CLAUSE FIVE – INSURANCE: The BORROWER must contract mandatory insurance for the


In [18]:
# All lines with "shall" being treated as obligations
def extract_obligations(text):

    lines = text.split("\n")

    obligations = []

    for line in lines:

        line = line.strip()

        if "shall" in line.lower() or "must" in line.lower():

            obligations.append(line)

    return list(dict.fromkeys(obligations))

In [19]:
extract_obligations(text)

['CLAUSE TWO – TERM: The financing period shall be up to 60 months, unless otherwise agreed',
 'CLAUSE THREE – PAYMENT: The BORROWER shall pay monthly installments, increased by',
 'CLAUSE FIVE – INSURANCE: The BORROWER must contract mandatory insurance for the']

In [20]:
# Improved Obligations function
def extract_obligations(text):

    lines = text.split("\n")

    obligations = []

    for line in lines:

        line = line.strip()

        if ("shall" in line.lower() or "must" in line.lower()):

            if ("BORROWER" in line or
                "LENDER" in line or
                "BANK" in line):

                obligations.append(line)

    return list(dict.fromkeys(obligations))

In [21]:
extract_obligations(text)

['CLAUSE THREE – PAYMENT: The BORROWER shall pay monthly installments, increased by',
 'CLAUSE FIVE – INSURANCE: The BORROWER must contract mandatory insurance for the']

In [22]:
def extract_contract_info(text):

    lines = text.split("\n")

    contract_type = lines[0].strip()

    borrower = None
    lender = None
    amount = None

    for line in lines:

        if "BORROWER:" in line:
            borrower = line.split("BORROWER:")[1].split(",")[0].strip()

        if "LENDER:" in line:
            lender = line.split("LENDER:")[1].split(",")[0].strip()

        if "FINANCED AMOUNT:" in line:
            amount = line.split("FINANCED AMOUNT:")[1].strip()

    clauses = []

    for line in lines:

        if "CLAUSE" in line and "–" in line:

            try:
                clause_name = line.split("–")[1].split(":")[0].strip()
                clauses.append(clause_name)

            except:
                pass

    clauses = list(dict.fromkeys(clauses))

    jurisdiction = None
        
    for line in lines:
        if "Courts of" in line:
            jurisdiction = line.split("Courts of")[1].split(", to")[0].strip()
            break

    obligations = extract_obligations(text)

    return {
        "contract_type": contract_type,
        "borrower": borrower,
        "lender": lender,
        "amount": amount,
        "clauses": clauses,
        "jurisdiction": jurisdiction,
        "obligations": obligations
    }

In [23]:
contract_info = extract_contract_info(text)

print(contract_info)

{'contract_type': 'STUDENT LOAN AGREEMENT', 'borrower': 'Michael Lewis', 'lender': 'FINANCIAL BANK OF AMERICA Inc.', 'amount': '$60,064.00.', 'clauses': ['PURPOSE', 'TERM', 'PAYMENT', 'GUARANTEES', 'INSURANCE', 'DEFAULT', 'TERMINATION', 'JURISDICTION'], 'jurisdiction': 'New York, NY', 'obligations': ['CLAUSE THREE – PAYMENT: The BORROWER shall pay monthly installments, increased by', 'CLAUSE FIVE – INSURANCE: The BORROWER must contract mandatory insurance for the']}


In [24]:
#re-run
all_contracts = []

import pdfplumber

for pdf_file in pdfs:

    text = ""

    with pdfplumber.open(pdf_file) as pdf:

        for page in pdf.pages:

            page_text = page.extract_text()

            if page_text:
                text += page_text + "\n"

    contract_info = extract_contract_info(text)

    contract_info["file"] = pdf_file.name

    all_contracts.append(contract_info)

print("Contracts processed:", len(all_contracts))

Contracts processed: 30


In [25]:
# Updated Dataframe
df = pd.DataFrame(all_contracts)
df.head()

,contract_type,borrower,lender,amount,clauses,jurisdiction,obligations,file
0,HEALTH FINANCING AGREEMENT,Julia Miller,FINANCIAL BANK OF AMERICA Inc.,"$42,751.00.","[PURPOSE, TERM, PAYMENT, GUARANTEES, INSURANCE...","New York, NY",[CLAUSE THREE – PAYMENT: The BORROWER shall pa...,Contract_16.pdf
1,MOTORCYCLE FINANCING AGREEMENT,Patricia Hall,FINANCIAL BANK OF AMERICA Inc.,"$36,160.00.","[PURPOSE, TERM, PAYMENT, GUARANTEES, INSURANCE...","New York, NY",[CLAUSE THREE – PAYMENT: The BORROWER shall pa...,Contract_4.pdf
2,HEALTH FINANCING AGREEMENT,Patricia Hall,FINANCIAL BANK OF AMERICA Inc.,"$40,267.00.","[PURPOSE, TERM, PAYMENT, GUARANTEES, INSURANCE...","New York, NY",[CLAUSE THREE – PAYMENT: The BORROWER shall pa...,Contract_5.pdf
3,STUDENT LOAN AGREEMENT,Richard Taylor,FINANCIAL BANK OF AMERICA Inc.,"$56,424.00.","[PURPOSE, TERM, PAYMENT, GUARANTEES, INSURANCE...","New York, NY",[CLAUSE THREE – PAYMENT: The BORROWER shall pa...,Contract_17.pdf
4,HEALTH FINANCING AGREEMENT,Mary Johnson,FINANCIAL BANK OF AMERICA Inc.,"$48,720.00.","[PURPOSE, TERM, PAYMENT, GUARANTEES, INSURANCE...","New York, NY",[CLAUSE THREE – PAYMENT: The BORROWER shall pa...,Contract_7.pdf


In [26]:
df.columns

Index(['contract_type', 'borrower', 'lender', 'amount', 'clauses',
       'jurisdiction', 'obligations', 'file'],
      dtype='object')

In [27]:
df[["contract_type", "obligations"]].head()

,contract_type,obligations
0,HEALTH FINANCING AGREEMENT,[CLAUSE THREE – PAYMENT: The BORROWER shall pa...
1,MOTORCYCLE FINANCING AGREEMENT,[CLAUSE THREE – PAYMENT: The BORROWER shall pa...
2,HEALTH FINANCING AGREEMENT,[CLAUSE THREE – PAYMENT: The BORROWER shall pa...
3,STUDENT LOAN AGREEMENT,[CLAUSE THREE – PAYMENT: The BORROWER shall pa...
4,HEALTH FINANCING AGREEMENT,[CLAUSE THREE – PAYMENT: The BORROWER shall pa...


### I built a rule-based contract intelligence pipeline that extracts agreement metadata, legal clauses, jurisdiction information, and contractual obligations from synthetic financing agreements. The extracted information was transformed into a structured dataset for downstream analysis and potential ML applications.

In [28]:
from pathlib import Path

Path("../data/processed").mkdir(exist_ok=True)

# Dataset Export

In [29]:
df.to_csv("../data/processed/contracts.csv", index=False)

df.to_excel("../data/processed/contracts.xlsx", index=False)

# Conclusions


A complete rule-based contract intelligence pipeline was developed for processing synthetic financing agreements.

The system successfully extracted:

- Contract Metadata
- Legal Clauses
- Jurisdiction Information
- Contractual Obligations

and transformed the information into a structured dataset.

---

## Key Outcomes

- Processed 30 PDF contracts

- Extracted structured legal information

- Built a reusable information extraction pipeline

- Created machine-readable datasets

- Prepared data for exploratory analysis and future ML applications

---

## Future Improvements

Potential extensions include:

- Named Entity Recognition (NER)
- Machine Learning-based Information Extraction
- Clause Classification
- Risk Assessment Models
- Contract Summarization
- LLM-powered Legal Analysis

---

## Skills Demonstrated

- Python Programming
- Document Processing
- Information Extraction
- Legal NLP
- Data Engineering
- ETL Pipelines
- Pandas
- Data Analysis